# Mistral Document AI through Azure AI Gateway

This notebook sends the same Mistral OCR request as `mistral_ocr_demo.ipynb`, but routes it through Azure API Management (APIM).

There are two separate authentication hops:

1. **Client to APIM**: the configured gateway key header authenticates the notebook to the gateway. Integrated Foundry AI Gateway commonly requests `api-key`.
2. **APIM to Foundry**: either the notebook's Microsoft Entra bearer token passes through APIM, or an APIM policy uses managed identity and injects the backend token.

A gateway-level `401 Access Denied` usually means the APIM subscription key is missing, invalid, or not authorized for the API.

## 1. Prerequisites

1. Add the Mistral OCR API to APIM with the operation path `/providers/mistral/azure/ocr`.
2. Record the gateway API base URL, including the API path suffix, for example `https://<gateway>.azure-api.net/<api-path>`.
3. Create or select an APIM subscription that can access the API and record one of its keys.
4. Choose the APIM-to-Foundry authentication pattern described below.
5. Copy `.env.example` to `.env`, fill in the gateway values, run `az login`, and place a document in `documents`.

Run `uv sync`, followed by `uv run jupyter lab`, from the repository root.

## 2. Choose the backend authentication mode

- `passthrough_entra` (default): the notebook gets a token for `https://ai.azure.com/.default`; APIM forwards the `Authorization` header unchanged. The signed-in user needs the **Cognitive Services User** role on the Foundry resource.
- `gateway_managed_identity`: the notebook omits `Authorization`. APIM must use an `authentication-managed-identity` policy for `https://ai.azure.com`, and the APIM managed identity needs the **Cognitive Services User** role on the Foundry resource.

The APIM subscription key is still sent in both modes. Never put either key directly in the notebook.

In [ ]:
import base64
import json
import mimetypes
import os
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

## 3. Load and validate configuration

`AZURE_AI_GATEWAY_ENDPOINT` is the APIM API base URL, not the Foundry project endpoint and not only the APIM hostname. It must include the API path suffix but omit `/providers/mistral/azure/ocr`.

In [ ]:
GATEWAY_ENDPOINT = os.getenv("AZURE_AI_GATEWAY_ENDPOINT", "").rstrip("/")
GATEWAY_SUBSCRIPTION_KEY = os.getenv("AZURE_AI_GATEWAY_SUBSCRIPTION_KEY", "")
GATEWAY_KEY_HEADER = os.getenv("AZURE_AI_GATEWAY_KEY_HEADER", "api-key")
OCR_DEPLOYMENT = os.getenv("AZURE_MISTRAL_OCR_DEPLOYMENT", "")
AUTH_MODE = os.getenv("AZURE_AI_GATEWAY_AUTH_MODE", "passthrough_entra")
OCR_ENDPOINT = f"{GATEWAY_ENDPOINT}/providers/mistral/azure/ocr"

required_values = {
    "AZURE_AI_GATEWAY_ENDPOINT": GATEWAY_ENDPOINT,
    "AZURE_AI_GATEWAY_SUBSCRIPTION_KEY": GATEWAY_SUBSCRIPTION_KEY,
    "AZURE_MISTRAL_OCR_DEPLOYMENT": OCR_DEPLOYMENT,
}
missing = [name for name, value in required_values.items() if not value]
if missing:
    raise RuntimeError(f"Set the following values in .env: {', '.join(missing)}")

parsed_endpoint = urlparse(GATEWAY_ENDPOINT)
if parsed_endpoint.scheme != "https" or not parsed_endpoint.netloc or not parsed_endpoint.path.strip("/"):
    raise ValueError(
        "AZURE_AI_GATEWAY_ENDPOINT must be an HTTPS APIM URL including the API path suffix, "
        "for example https://<gateway>.azure-api.net/<api-path>."
    )

supported_auth_modes = {"passthrough_entra", "gateway_managed_identity"}
if AUTH_MODE not in supported_auth_modes:
    raise ValueError(f"AZURE_AI_GATEWAY_AUTH_MODE must be one of {sorted(supported_auth_modes)}")

print(f"Gateway API: {GATEWAY_ENDPOINT}")
print(f"OCR deployment: {OCR_DEPLOYMENT}")
print(f"Backend authentication: {AUTH_MODE}")
print(f"Gateway key header: {GATEWAY_KEY_HEADER}")
print("APIM subscription key loaded (value hidden).")

## 4. Build request headers

This cell fails early if local Microsoft Entra authentication is required but unavailable. The token and subscription key are never displayed.

In [ ]:
TOKEN_SCOPE = "https://ai.azure.com/.default"
credential = DefaultAzureCredential() if AUTH_MODE == "passthrough_entra" else None


def gateway_headers() -> dict[str, str]:
    headers = {
        "Content-Type": "application/json",
        GATEWAY_KEY_HEADER: GATEWAY_SUBSCRIPTION_KEY,
    }
    if credential is not None:
        token = credential.get_token(TOKEN_SCOPE)
        headers["Authorization"] = f"Bearer {token.token}"
    return headers


if credential is not None:
    _ = credential.get_token(TOKEN_SCOPE)
    print("Microsoft Entra token acquired for Foundry passthrough.")
else:
    print("APIM managed identity mode selected; the gateway must add backend authentication.")

## 5. Select and encode a document

Update `DOCUMENT_PATH` if needed. The file is encoded as a data URL, matching the direct Foundry demo.

In [ ]:
DOCUMENT_PATH = Path("documents/AdventureWorksPO_HROrder.png")


def document_data_url(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(f"Document not found: {path.resolve()}")

    media_type, _ = mimetypes.guess_type(path.name)
    if media_type is None:
        raise ValueError(f"Could not determine the media type for {path.name}")

    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{encoded}"


print(f"Document ready: {DOCUMENT_PATH.is_file()}")

## 6. Send the OCR request through APIM

Error output includes the status, APIM request identifier, and a bounded response body. It does not print request headers or credentials.

In [ ]:
def raise_gateway_error(response: requests.Response) -> None:
    if response.ok:
        return

    request_id = response.headers.get("apim-request-id", response.headers.get("x-ms-request-id", "unavailable"))
    response_body = response.text[:2000]
    hint = ""
    if response.status_code == 401:
        hint = " Check the APIM subscription key and confirm its subscription can access this API."
    elif response.status_code == 403:
        hint = " Check APIM policies and the identity/RBAC used for the Foundry backend."
    elif response.status_code == 404:
        hint = " Check the APIM API path suffix and OCR operation path."

    raise RuntimeError(
        f"AI Gateway request failed with HTTP {response.status_code}. "
        f"Request ID: {request_id}.{hint}\nResponse body:\n{response_body}"
    )


def read_document(path: Path, *, include_images: bool = False) -> dict[str, Any]:
    payload = {
        "model": OCR_DEPLOYMENT,
        "document": {
            "type": "document_url",
            "document_url": document_data_url(path),
        },
        "include_image_base64": include_images,
    }

    response = requests.post(
        OCR_ENDPOINT,
        headers=gateway_headers(),
        json=payload,
        timeout=180,
    )
    raise_gateway_error(response)
    return response.json()


ocr_result = read_document(DOCUMENT_PATH)
print(json.dumps(ocr_result, indent=2)[:5000])

## 7. Inspect the result

In [ ]:
def summarize_result(result: dict[str, Any]) -> dict[str, Any]:
    pages = result.get("pages", [])
    return {
        "page_count": len(pages),
        "markdown_characters": sum(len(page.get("markdown", "")) for page in pages),
        "image_count": sum(len(page.get("images", [])) for page in pages),
    }


display(summarize_result(ocr_result))

pages = ocr_result.get("pages", [])
if pages:
    display(Markdown(pages[0].get("markdown", "")))
else:
    print("The OCR response did not contain any pages.")

## Troubleshooting map

| Result | Likely cause | Action |
|---|---|---|
| `401 Access Denied` from APIM | Missing or invalid APIM subscription key | Verify `AZURE_AI_GATEWAY_SUBSCRIPTION_KEY` and API/product access |
| `401` mentioning a bearer token | Backend token missing, rejected, or wrong audience | Verify the selected auth mode and APIM backend-auth policy |
| `403` from Foundry | Backend identity lacks permission | Grant **Cognitive Services User** on the Foundry resource |
| `404` | Incorrect gateway API suffix or operation path | Copy the API URL from APIM and verify `/providers/mistral/azure/ocr` is imported |
| `429` | APIM or backend quota/rate limit | Inspect APIM policies and Foundry deployment quota |

For APIM tracing, enable tracing for a test request in the APIM portal and use the request ID emitted by this notebook.